# Entrega 2 — Parte C: El puente hacia la Entrega Final

Del plan óptimo de la Parte B se **extrae y congela el programa de la semana 12** (lunes 14/12/2026):

- qué SKU se producen, en qué cantidad y en qué línea;
- cuántas horas de preparación supuso el modelo en cada recurso.

Ese último número es el que la Entrega Final va a poner a prueba. Allí la secuencia real, con la
matriz CIP asimétrica, dirá si las horas reservadas alcanzaban.

Este notebook **no vuelve a resolver** el modelo. Lee la solución guardada por la Parte B en
`resultados/solucion_base.json`, para que el programa congelado sea exactamente el del plan
reportado.

In [1]:
# Preparación del entorno: ubica el paquete `entrega2` (carpeta "Entrega 2") y lo importa.
import sys
from pathlib import Path

carpeta = Path.cwd()
while not (carpeta / "entrega2").is_dir() and carpeta != carpeta.parent:
    carpeta = carpeta.parent
sys.path.insert(0, str(carpeta))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp as pl

from entrega2 import parametros as pa, modelo as mo, analisis as an, heuristicas as he

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pa.RESULTADOS.mkdir(parents=True, exist_ok=True)
pa.FIGURAS.mkdir(parents=True, exist_ok=True)
print("Paquete cargado desde:", carpeta)
print("PuLP", pl.__version__, "| solvers disponibles:", pl.listSolvers(onlyAvailable=True))

P = pa.construir_parametros(pa.Escenario())
sol = an.cargar_solucion(pa.RESULTADOS / "solucion_base.json")
assert abs(an.costos(P, sol)["Total"] - float(pd.read_csv(pa.RESULTADOS / "resolucion_base.csv", index_col=0)
                                                .loc["Valor objetivo (CLP)"].iloc[0].replace(",", ""))) < 1.0
print("Solución base cargada y verificada contra el valor objetivo de la Parte B.")
T12 = pa.SEMANA_FOCAL
print("Semana focal:", T12, "| lunes", P["fecha"][T12])

Paquete cargado desde: /home/claude/gdbs2/Entrega 2
PuLP 2.9.0 | solvers disponibles: ['PULP_CBC_CMD']
Solución base cargada y verificada contra el valor objetivo de la Parte B.
Semana focal: 12 | lunes 2026-12-14


## 1. Programa de producción de la semana 12

Para cada SKU se informa la producción **bruta**, que es lo que la línea debe envasar, y la
**vendible**, que es lo que queda después de la merma heredada de la Entrega 1. También se muestra
la cartera en firme que esa producción debe cubrir.

In [2]:
com = pa.cargar_datos()["compromisos"]
filas = []
for i in P["SKUS"]:
    x = sol["x"][(i, T12)]
    filas.append({
        "SKU": i, "Familia": P["familia"][i], "Línea": P["linea"][i],
        "Producción bruta (cajas)": round(x),
        "Producción vendible (cajas)": round(P["rend"][i] * x),
        "Cartera en firme (cajas)": P["C"][i],
        "Órdenes": int((com.sku == i).sum()),
        "Holgura sobre cartera (cajas)": round(P["rend"][i] * x - P["C"][i]),
        "Horas de envasado": x / P["tasa"][i],
        "Litros de jarabe (mezclados en sem. 11)": P["litros_ef"][i] * x,
    })
programa = pd.DataFrame(filas).set_index("SKU").sort_values(["Línea", "Familia"])
display(programa.round(1))
assert (programa["Holgura sobre cartera (cajas)"] >= -0.5).all(), "La producción no cubre la cartera en firme."
programa.to_csv(pa.RESULTADOS / "programa_semana12.csv")
print("Los 9 SKU cubren su cartera en firme. Total bruto:", f"{programa['Producción bruta (cajas)'].sum():,.0f} cajas.")

,Familia,Línea,Producción bruta (cajas),Producción vendible (cajas),Cartera en firme (cajas),Órdenes,Holgura sobre cartera (cajas),Horas de envasado,Litros de jarabe (mezclados en sem. 11)
SKU,,,,,,,,,
CIT-L05,CIT,L1,855,839,839.00,2,0,4.80,"5,191.40"
FUN-B05,FUN,L1,2606,2556,"1,340.00",2,1216,14.50,"15,815.80"
NEC-D05,NEC,L1,2712,2660,898.00,2,1762,15.10,"16,460.70"
CIT-L15,CIT,L2,3793,3675,"1,026.00",2,2649,18.10,"34,246.70"
CIT-L30,CIT,L2,217,210,157.00,1,53,1.70,"2,606.20"
FUN-B15,FUN,L2,2546,2467,860.00,2,1607,12.10,"22,987.80"
FUN-B30,FUN,L2,68,66,66.00,1,0,0.50,819.80
NEC-D15,NEC,L2,1736,1682,"1,320.00",2,362,8.30,"15,675.30"
NEC-D30,NEC,L2,323,313,313.00,1,0,2.50,"3,887.60"


Los 9 SKU cubren su cartera en firme. Total bruto: 14,856 cajas.


## 2. Horas de preparación supuestas por el modelo

En la semana 12 el modelo reservó, por cada campaña de familia abierta en una línea, el tiempo
esperado de entrada $\tau_f$. En L2 agregó 0,6 h por cada cambio de formato dentro de una familia. En
la mezcladora reservó 1,2 h de CIP por cada familia mezclada en la semana 11.

In [3]:
prep = []
for l in P["LINEAS"]:
    fams = [f for f in P["FAMILIAS"] if sol["w"].get((l, f, T12), 0) > 0.5]
    cip = sum(P["tau_cip"][f] for f in fams)
    fmt = sum(P["horas_formato"]["L2"] * sol["fc"][(f, T12)] for f in P["FAMILIAS"]) if l == "L2" else 0.0
    prod = sum(sol["x"][(i, T12)] / P["tasa"][i] for i in P["skus_linea"][l])
    prep.append({"Recurso": l, "Familias activas": ", ".join(fams), "Horas CIP supuestas (Σ τ_f)": cip,
                 "Cambios de formato": sum(sol["fc"][(f, T12)] for f in P["FAMILIAS"]) if l == "L2" else 0,
                 "Horas de cambio de formato": fmt, "Horas de preparación totales": cip + fmt,
                 "Horas de producción": prod, "Horas regulares": P["H"][(l, T12)],
                 "Horas extra planificadas": sol["E"][(l, T12)]})
s11 = T12 - 1
fams_m = [f for f in P["FAMILIAS"] if sol["z"][(f, s11)] > 0.5]
prep.append({"Recurso": "Mezcla (sem. 11)", "Familias activas": ", ".join(fams_m),
             "Horas CIP supuestas (Σ τ_f)": P["cip_mezcla"][s11] * len(fams_m), "Cambios de formato": 0,
             "Horas de cambio de formato": 0.0, "Horas de preparación totales": P["cip_mezcla"][s11] * len(fams_m),
             "Horas de producción": sum(sol["V"][(f, s11)] for f in P["FAMILIAS"]) / P["tasa_mezcla"][s11],
             "Horas regulares": P["Hm"][s11], "Horas extra planificadas": sol["Em"][s11]})
preparacion = pd.DataFrame(prep).set_index("Recurso")
preparacion["Holgura (h)"] = (preparacion["Horas regulares"] + preparacion["Horas extra planificadas"]
                              - preparacion["Horas de producción"] - preparacion["Horas de preparación totales"])
display(preparacion.round(2))
preparacion.to_csv(pa.RESULTADOS / "preparacion_semana12.csv")
for rec, fila in preparacion.iterrows():
    holgura = max(0.0, round(fila["Holgura (h)"], 2))
    print(f"{rec}: el modelo supuso {fila['Horas de preparación totales']:.1f} h de preparación "
          f"y deja {holgura:.1f} h de holgura" + (" (recurso lleno: cualquier hora adicional de limpieza exige horas extra)." if holgura < 0.5 else "."))

,Familias activas,Horas CIP supuestas (Σ τ_f),Cambios de formato,Horas de cambio de formato,Horas de preparación totales,Horas de producción,Horas regulares,Horas extra planificadas,Holgura (h)
Recurso,,,,,,,,,
L1,"CIT, FUN, NEC",5.70,0.00,0.00,5.70,34.30,40.00,0.00,-0.00
L2,"CIT, FUN, NEC",5.70,3.00,1.80,7.50,43.13,48.00,2.63,-0.00
Mezcla (sem. 11),"CIT, FUN, NEC",3.60,0.00,0.00,3.60,51.17,55.00,0.00,0.23


L1: el modelo supuso 5.7 h de preparación y deja 0.0 h de holgura (recurso lleno: cualquier hora adicional de limpieza exige horas extra).
L2: el modelo supuso 7.5 h de preparación y deja 0.0 h de holgura (recurso lleno: cualquier hora adicional de limpieza exige horas extra).
Mezcla (sem. 11): el modelo supuso 3.6 h de preparación y deja 0.2 h de holgura (recurso lleno: cualquier hora adicional de limpieza exige horas extra).


## 3. Lo que la Entrega Final pondrá a prueba

El tiempo supuesto es un **promedio**. La secuencia real depende del orden en que se envasen las
familias y de cuál quedó en la línea al cerrar la semana 11. Como adelanto del contraste que pide la
Entrega Final (sección 8.2), se calcula el rango de horas CIP reales posibles para las familias
activas de cada línea:

- la **mejor** secuencia, partiendo de la familia que convenga;
- la **peor** secuencia.

Las dos incluyen la limpieza de entrada desde la familia con que terminó la semana anterior.

In [4]:
import itertools
mat = pa.cargar_datos()["setups"].set_index(["familia_origen", "familia_destino"]).horas_limpieza_cip
contraste = []
for l in P["LINEAS"]:
    fams = [f for f in P["FAMILIAS"] if sol["w"].get((l, f, T12), 0) > 0.5]
    supuesto = sum(P["tau_cip"][f] for f in fams)
    opciones = []
    for previa in P["FAMILIAS"]:
        for orden in itertools.permutations(fams):
            ruta = [previa] + list(orden)
            opciones.append(sum(mat[(a, b)] for a, b in zip(ruta, ruta[1:])))
    contraste.append({"Línea": l, "Familias activas": len(fams), "CIP supuesto por el modelo (h)": supuesto,
                      "CIP real, mejor secuencia (h)": min(opciones), "CIP real, peor secuencia (h)": max(opciones)})
contraste = pd.DataFrame(contraste).set_index("Línea")
display(contraste.round(2))
contraste.to_csv(pa.RESULTADOS / "contraste_preliminar_cip_semana12.csv")
for l, f in contraste.iterrows():
    print(f"{l}: el modelo reservó {f['CIP supuesto por el modelo (h)']:.1f} h; la secuencia real puede necesitar "
          f"entre {f['CIP real, mejor secuencia (h)']:.1f} y {f['CIP real, peor secuencia (h)']:.1f} h.")

,Familias activas,CIP supuesto por el modelo (h),"CIP real, mejor secuencia (h)","CIP real, peor secuencia (h)"
Línea,,,,
L1,3,5.70,2.40,7.00
L2,3,5.70,2.40,7.00


L1: el modelo reservó 5.7 h; la secuencia real puede necesitar entre 2.4 y 7.0 h.
L2: el modelo reservó 5.7 h; la secuencia real puede necesitar entre 2.4 y 7.0 h.


**Lectura.** El modelo táctico reservó un promedio. Si la Entrega Final consigue una secuencia cercana
a la mejor (la "campaña natural" del enunciado), sobra capacidad. Si las órdenes con fecha
comprometida obligan a un orden desfavorable, falta tiempo, y la sección 2 muestra cuánta holgura
queda en cada línea para absorberlo. Cuando un recurso no tiene holgura, cada hora de limpieza
adicional se paga con horas extra o desplaza producción. Medir
esa brecha y volver a resolver la Entrega 2 con la capacidad corregida es el cierre del ciclo
(sección 8.2).

## 4. Tabla de traspaso hacia la Entrega Final

Archivos que la Entrega Final usará **literalmente** como entrada:

| Archivo | Contenido |
| :--- | :--- |
| `resultados/programa_semana12.csv` | SKU, línea, producción bruta y vendible, cartera, horas de envasado y litros de jarabe |
| `resultados/preparacion_semana12.csv` | Horas de preparación supuestas por recurso (**el número a contrastar**) |
| `resultados/contraste_preliminar_cip_semana12.csv` | Rango de CIP real posible con las familias activas |
| `datos/compromisos_semana_focal.csv` | Órdenes con día, hora, prioridad y multa (sin cambios) |

In [5]:
ordenes = com.merge(programa[["Producción vendible (cajas)"]], left_on="sku", right_index=True)
display(ordenes[["orden", "sku", "linea_envasado", "cliente", "cantidad_comprometida_cajas", "dia_comprometido",
                 "hora_comprometida", "peso_prioridad", "multa_atraso_clp_h"]].sort_values(["linea_envasado", "dia_comprometido"]))
print("Archivos de traspaso escritos en", pa.RESULTADOS.relative_to(pa.RAIZ))

,orden,sku,linea_envasado,cliente,cantidad_comprometida_cajas,dia_comprometido,hora_comprometida,peso_prioridad,multa_atraso_clp_h
0,OP-001,NEC-D05,L1,Mayorista Sur,389,2,14:00,1,26000
11,OP-012,FUN-B05,L1,Cadena Retail A,564,2,10:00,5,130000
5,OP-006,CIT-L05,L1,CD Santiago,347,4,18:00,3,78000
6,OP-007,CIT-L05,L1,Cadena Retail B,492,4,18:00,4,104000
1,OP-002,NEC-D05,L1,CD Santiago,509,5,18:00,3,78000
10,OP-011,FUN-B05,L1,CD Santiago,776,5,18:00,3,78000
13,OP-014,FUN-B15,L2,CD Santiago,459,2,14:00,3,78000
3,OP-004,NEC-D15,L2,Cadena Retail A,736,3,18:00,5,130000
4,OP-005,NEC-D30,L2,CD Santiago,313,3,18:00,3,78000
7,OP-008,CIT-L15,L2,Mayorista Sur,472,3,18:00,1,26000


Archivos de traspaso escritos en Entrega 2/resultados
